In [1]:
import os 
from os import listdir
from os.path import join

In [2]:
original_cases_path = '/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train'
original_cases_path_D = '/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train_D'

inferences_path = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered/VS-DDPM_Task1_2_1000_timestep__patchsize_2_Unet_MAE_SSIM_tahn_AFP_128_128_32_distinctNorm_region_HN_TH_AB_linear_pen_var_random_T_CTminmax-1000_1600_finetune_new_NormalizeIntensityd_Scaled_adjustStep_adjust_overlap__T__599/constant/"

In [ ]:
patients_list = []

for region in listdir(inferences_path):
    region_path = join(inferences_path, region)
    for patient_id in listdir(region_path):
        if not patient_id.endswith('.json'):
            patient_id_path = join(region_path, patient_id)
            institution = patient_id[3]

            pred_path = join(patient_id_path, 'ct_pred.mha')
        
            if institution=="D":
                mri_path = join(original_cases_path_D, 'Task1', region, patient_id, 'mr.mha')
            else:
                mri_path = join(original_cases_path, 'Task1', region, patient_id, 'mr.mha')

            patients_list.append(
                {
                    'pred': pred_path,
                    'mri': mri_path

                }
            )

patients_list[0]


In [6]:
import os
import numpy as np
import SimpleITK as sitk
# Output folder
output_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_1000_timestep_25_patchsize_2_SwinVIT_MSE_MAE_SSIM_128_128_32_distinctNorm_region_HN_TH_AB_DA_0.5_new_NormalizeIntensityd_Scaled_2000_-1Clip/constant"

os.makedirs(output_folder, exist_ok=True)

# Process each pair
def process_pair(mri_path, pred_path, output_path):
    mri_img = sitk.ReadImage(mri_path)
    pred_img = sitk.ReadImage(pred_path)

    mri_array = sitk.GetArrayFromImage(mri_img)
    pred_array = sitk.GetArrayFromImage(pred_img)

    pred_array[mri_array == 0] = -1000

    new_pred_img = sitk.GetImageFromArray(pred_array)
    new_pred_img.CopyInformation(pred_img)
    sitk.WriteImage(new_pred_img, output_path)

# Apply to all data
for entry in patients_list:
    pred_path = entry['pred']
    mri_path = entry['mri']

    patient_id = pred_path.split('/')[-2]
    region = pred_path.split('/')[-3]

    output_folder_here = join(output_folder, region, patient_id)
    os.makedirs(output_folder_here, exist_ok=True)

    output_path = os.path.join(output_folder_here, f"ct_pred.mha")

    process_pair(mri_path, pred_path, output_path)

print("Processing complete.")